In [3]:
!pip install -U pip
!pip install flwr==1.30.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [18]:
!pip install -U flwr-datasets

In [23]:
import flwr

print(flwr.__version__)

1.30.0


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision.transforms import Compose, Normalize, ToTensor

from flwr.app import ArrayRecord, Context, ConfigRecord, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp
from flwr.serverapp import Grid, ServerApp
from flwr.serverapp.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner

from functools import partial

from datasets import load_dataset


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [9]:
client_app = ClientApp()


@client_app.train()
def train(msg: Message, context: Context):
    """Train locally (SGD), convert delta to an 'effective gradient', send to server."""

    # 1) Build model
    if context.run_config["model-type"] == "bn":
        print("Client: Using BatchNorm model")
        model = NetBN()
    else:
        print("Client: Using standard model")
        model = Net()

    # 2) Load global weights from server
    model.load_state_dict(msg.content["arrays"].to_torch_state_dict(), strict=False)
    model.to(DEVICE)

    # 3) Load local data
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    trainloader, _ = load_data(partition_id, num_partitions)

    # 4) Local train (SGD) and get delta = w_local - w_global for PARAMETERS
    local_epochs = int(context.run_config["local-epochs"])
    lr_local = float(msg.content["config"]["lr"])  # comes from server config

    print(10 * "=" + f" LOCAL TRAIN (delta->grad), lr_local={lr_local} " + 10 * "=")

    gradient, avg_loss, steps = compute_gradients(
        model=model,
        trainloader=trainloader,
        epochs=local_epochs,
        lr=lr_local,
        device=DEVICE,
    )

    # 6) Send to server
    content = RecordDict(
        {
            "arrays": ArrayRecord(gradient),
            "metrics": ArrayRecord(
                {
                    "train_loss": float(avg_loss),
                    "num-examples": len(trainloader.dataset),
                    "steps": int(steps),
                    "lr_local": lr_local,
                }
            ),
        }
    )
    return Message(content=content, reply_to=msg)


@client_app.evaluate()
def evaluate(msg: Message, context: Context):
    """Evaluate the model on local validation data."""
    print(10 * "=" + " LOCAL EVALUATE " + 10 * "=")

    # Build model
    if context.run_config["model-type"] == "bn":
        print("Client eval: Using BatchNorm model")
        model = NetBN()
    else:
        print("Client eval: Using standard model")
        model = Net()

    # Load weights
    model.load_state_dict(msg.content["arrays"].to_torch_state_dict(), strict=False)
    model.to(DEVICE)

    # Load local validation data
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    _, valloader = load_data(partition_id, num_partitions)

    # Evaluate
    eval_loss, eval_acc = test(model, valloader, device)

    # Reply
    metrics = {
        "eval_loss": float(eval_loss),
        "eval_acc": float(eval_acc),
        "num-examples": len(valloader.dataset),
    }
    content = RecordDict({"metrics": MetricRecord(metrics)})
    return Message(content=content, reply_to=msg)


In [21]:
def project(self, grad):
    flat=torch.cat([g.flatten()
            for g in grad.values()])

    norm=torch.norm(flat)
    if norm<=self.radius:
        return grad

    coef=self.radius/norm
    return {
        k:v*coef
        for k,v in grad.items()
    }

def aggregate_gradients(self, gradients):
    grads=[]
    for g in gradients:
        if self.project_each_client:
            g=self.project(g)
        grads.append(g)
    agg={}
    for key in grads[0]:
        agg[key]=sum(
            g[key]
            for g in grads
        )/len(grads)

    if self.project_agg:
        agg=self.project(agg)

    return agg

class GradientProjection(FedAvg):

    def __init__(
        self,
        lr=1e-2,
        radius=1.0,
        project_each_client=True,
        project_agg=True,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.lr=lr
        self.radius=radius
        self.project_each_client=project_each_client
        self.project_agg=project_agg

In [13]:
# inne metody uśredniania
from flwr.serverapp.strategy import FedAvg
from flwr.serverapp.strategy import FedAdam
from flwr.serverapp.strategy import FedYogi
from flwr.serverapp.strategy import FedProx

def create_fedprox(context):
    return FedProx(
        fraction_train=context.run_config["fraction-train"],
        proximal_mu=0.01,
    )

def create_fedyogi(context):
    return FedYogi(
        fraction_train=context.run_config["fraction-train"],
        eta=0.001,
        eta_l=1.0,
        beta_1=0.9,
        beta_2=0.99,
        tau=1e-3,
    )

def create_fedavg(context):
    return FedAvg(
        fraction_train=context.run_config["fraction-train"],
    )

def create_fedadam(context):
    return FedAdam(
        fraction_train=context.run_config["fraction-train"],
        eta=0.001,
        eta_l=1.0,
        beta_1=0.9,
        beta_2=0.99,
        tau=1e-9,
    )

def get_strategy(name, context):
    if name == "FedAvg":
        return create_fedavg(context)
    if name == "FedAdam":
        return create_fedadam(context)
    if name == "FedYogi":
        return create_fedyogi(context)
    if name == "FedProx":
        return create_fedprox(context)
    if name == "GradientProjection":
        return GradientProjection(
            lr=context.run_config["lr"],
            project_each_client=True,
            project_agg=True,
        )

    raise ValueError(f"Unknown strategy: {name}")

In [ ]:
def get_model_type(name, context):
    if name == "bn":
        return NetBN()

    return Net()

In [10]:
server_app = ServerApp()


@server_app.main()
def main(grid: Grid, context: Context) -> None:
    """Main entry point for the ServerApp."""

    # Read run config
    num_rounds: int = context.run_config["num-server-rounds"]
    lr: float = context.run_config["lr"]
    print(f"Starting training for {num_rounds} rounds with initial lr={lr}")

    # Load global model
    global_model = get_model_type(context.run_config["model-type"])
    strategy = get_strategy(context.run_config["strategy"])

    arrays = ArrayRecord(filter_state_dict(global_model.state_dict()))

    # Start strategy, run FedAvg for `num_rounds`
    result = strategy.start(
        grid=grid,
        initial_arrays=arrays,
        train_config=ConfigRecord({"lr": lr}),
        num_rounds=num_rounds,
        evaluate_fn=partial(global_evaluate, model_type=context.run_config["model-type"]),
    )

    # Save final model to disk
    print("\nSaving final model to disk...")
    state_dict = result.arrays.to_torch_state_dict()
    torch.save(state_dict, "final_model.pt")


def global_evaluate(server_round: int, arrays: ArrayRecord, model_type: str) -> MetricRecord:
    """Evaluate model on central data."""

    print(10 * '=' + ' GLOBAL EVALUATE ' + 10 * '=')

    # Load the model and initialize it with the received weights
    if model_type == "bn":
        print('Server Eval: Using BatchNorm model')
        model = NetBN()
    else:
        print('Server Eval: Using standard model')
        model = Net()

    model.load_state_dict(filter_state_dict(arrays.to_torch_state_dict()), strict=False)
    model.to(DEVICE)

    # BN running stats are not aggregated across clients, so switch BN layers to
    # train mode to use batch statistics during evaluation instead of stale defaults.
    if model_type == "bn":
        import torch.nn as nn
        for m in model.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                m.train()

    # Load entire test set
    test_dataloader = load_centralized_dataset()

    # Evaluate the global model on the test set
    test_loss, test_acc = test(model, test_dataloader, device)

    # Return the evaluation metrics
    return MetricRecord({"accuracy": test_acc, "loss": test_loss})


def filter_state_dict(state_dict):
    return {
        k: v for k, v in state_dict.items()
        if "num_batches_tracked" not in k
    }

In [11]:
"""flower-tutorial: A Flower / PyTorch app."""

class Net(nn.Module):
    """Model (simple CNN adapted from 'PyTorch: A 60 Minute Blitz')"""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class NetBN(nn.Module):
    """Simple CNN with BatchNorm"""

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 6, 5)
        self.bn1 = nn.BatchNorm2d(6)

        self.conv2 = nn.Conv2d(6, 16, 5)
        self.bn2 = nn.BatchNorm2d(16)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.bn3 = nn.BatchNorm1d(120)

        self.fc2 = nn.Linear(120, 84)
        self.bn4 = nn.BatchNorm1d(84)

        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn3(self.fc1(x)))
        x = F.relu(self.bn4(self.fc2(x)))
        return self.fc3(x)


fds = None  # Cache FederatedDataset

pytorch_transforms = Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])


def apply_transforms(batch):
    """Apply transforms to the partition from FederatedDataset."""
    batch["img"] = [pytorch_transforms(img) for img in batch["img"]]
    return batch


def load_data(partition_id: int, num_partitions: int):
    """Load partition CIFAR10 data."""
    # Only initialize `FederatedDataset` once
    global fds
    if fds is None:
        partitioner = IidPartitioner(num_partitions=num_partitions)
        fds = FederatedDataset(
            dataset="uoft-cs/cifar10",
            partitioners={"train": partitioner},
        )
    partition = fds.load_partition(partition_id)
    # Divide data on each node: 80% train, 20% test
    partition_train_test = partition.train_test_split(test_size=0.2, seed=42)
    # Construct dataloaders
    partition_train_test = partition_train_test.with_transform(apply_transforms)
    trainloader = DataLoader(partition_train_test["train"], batch_size=32, shuffle=True)
    testloader = DataLoader(partition_train_test["test"], batch_size=32)
    return trainloader, testloader

def load_centralized_dataset():
    """Load test set and return dataloader."""
    # Load entire test set
    test_dataset = load_dataset("uoft-cs/cifar10", split="test")
    dataset = test_dataset.with_format("torch").with_transform(apply_transforms)
    return DataLoader(dataset, batch_size=128)

def train(net, trainloader, epochs, lr, device):
    """Train the model on the training set."""
    net.to(device)  # move model to GPU if available
    criterion = torch.nn.CrossEntropyLoss().to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    net.train()
    running_loss = 0.0
    for _ in range(epochs):
        for batch in trainloader:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)
            optimizer.zero_grad()
            loss = criterion(net(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    avg_trainloss = running_loss / len(trainloader)
    return avg_trainloss


def test(net, testloader, device):
    """Validate the model on the test set."""
    net.to(device)
    net.eval()
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    with torch.no_grad():
        for batch in testloader:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)
            outputs = net(images)
            loss += criterion(outputs, labels).item()
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
    accuracy = correct / len(testloader.dataset)
    loss = loss / len(testloader)
    return loss, accuracy

def compute_gradients(net, trainloader, epochs, device):
    net.to(device)
    net.train()
    criterion = torch.nn.CrossEntropyLoss().to(device)

    grad_acc = {
        name: torch.zeros_like(param, device=device)
        for name, param in net.named_parameters()
    }

    total_loss = 0.0
    steps = 0

    for _ in range(epochs):
        for batch in trainloader:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)

            net.zero_grad(set_to_none=True)
            loss = criterion(net(images), labels)
            loss.backward()

            total_loss += loss.item()
            steps += 1

            for name, param in net.named_parameters():
                if param.grad is not None:
                    grad_acc[name] += param.grad.detach()

    for k in grad_acc:
        grad_acc[k] /= max(1, steps)

    avg_loss = total_loss / max(1, steps)
    return grad_acc, avg_loss, steps


def local_train_and_return_delta(model, trainloader, epochs, lr, device):
    model.to(device)
    model.train()
    criterion = torch.nn.CrossEntropyLoss().to(device)
    opt = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4,
    )

    # snapshot initial parameters
    w0 = {k: v.detach().clone() for k, v in model.state_dict().items()}

    total_loss = 0.0
    steps = 0

    for _ in range(epochs):
        for batch in trainloader:
            x = batch["img"].to(device)
            y = batch["label"].to(device)

            opt.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()

            total_loss += loss.item()
            steps += 1

    w1 = model.state_dict()

    # delta for parameters only (named_parameters keys)
    delta = {}
    for name, p in model.named_parameters():
        delta[name] = (w1[name] - w0[name]).detach()

    avg_loss = total_loss / max(1, steps)
    return delta, avg_loss, steps

In [ ]:
def run_training(
    num_supernodes=10,
    num_rounds=10,
    lr=0.01,
    model_type="standard",
    strategy="default"
):
    backend_config = {
        "client_resources": {
            "num_cpus": 1,
            "num_gpus": 0.0,
        }
    }

    run_config = {
        "num-server-rounds": num_rounds,
        "lr": lr,
        "model-type": model_type,
        "local-epochs": 1,
        "fraction-train": 1.0,
        "fraction-custom-train": 1.0,
        "fraction-custom-evaluate": 1.0,
        "strategy": "default",
    }

    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=num_supernodes,
        backend_config=backend_config,
        run_config=run_config,
    )